In [3]:
import pandas as pd
from datetime import date
from sqlalchemy import create_engine

# Setup database connection (using your existing configuration)
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

def calculate_performance_vs_index(start_date, end_date):
    """
    Calculates stock performance relative to the SET Index within a specific period.
    """
    
    # 1. Fetch SET Index data for the period
    index_sql = f"""
    SELECT date, setindex AS index_price 
    FROM setindex 
    WHERE date BETWEEN '{start_date}' AND '{end_date}'
    ORDER BY date ASC
    """
    df_index = pd.read_sql(index_sql, const)
    
    if df_index.empty:
        return "No Index data found for the specified period."

    # Calculate Index Return
    idx_start_val = df_index.iloc[0]['index_price']
    idx_end_val = df_index.iloc[-1]['index_price']
    index_pct_change = ((idx_end_val - idx_start_val) / idx_start_val) * 100

    # 2. Fetch Active Stocks from the buy table
    buy_sql = """
    SELECT name 
    FROM buy 
    WHERE active = 1
    """
    df_active_stocks = pd.read_sql(buy_sql, const)
    stock_names = df_active_stocks['name'].tolist()

    performance_results = []

    # 3. Calculate return for each stock and compare with Index
    for name in stock_names:
        stock_sql = f"""
        SELECT date, price 
        FROM price 
        WHERE name = '{name}' AND date BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY date ASC
        """
        df_stock_price = pd.read_sql(stock_sql, const)
        
        if not df_stock_price.empty:
            stk_start_val = df_stock_price.iloc[0]['price']
            stk_end_val = df_stock_price.iloc[-1]['price']
            stock_pct_change = ((stk_end_val - stk_start_val) / stk_start_val) * 100
            
            # Comparison Logic
            relative_perf = stock_pct_change - index_pct_change
            status = "Better" if relative_perf > 0 else "Worse"
            
            performance_results.append({
                'Name': name,
                'Stock_Return_%': stock_pct_change,
                'Index_Return_%': index_pct_change,
                'Relative_Perf_%': relative_perf,
                'Comparison': status
            })

    # Create Final DataFrame
    df_comparison = pd.DataFrame(performance_results)
    
    # Sort by performance relative to market
    return df_comparison.sort_values(by='Relative_Perf_%', ascending=False)

# Example Usage:
# Define your period (you can use your 'today' variable or specific dates)
target_start = '2023-01-01'
target_end = str(date.today())

comparison_report = calculate_performance_vs_index(target_start, target_end)

# Display results
print(f"Performance Comparison from {target_start} to {target_end}")
print(comparison_report)

# Optional: Export to CSV
# comparison_report.to_csv('stock_mar

Performance Comparison from 2023-01-01 to 2026-03-06
       Name  Stock_Return_%  Index_Return_%  Relative_Perf_% Comparison
6    ADVANC       95.372751       -8.976337       104.349087     Better
3       PTT       12.977099       -8.976337        21.953436     Better
4       MCS        5.555556       -8.976337        14.531892     Better
8     WHART        3.703704       -8.976337        12.680040     Better
26      RCL        0.819672       -8.976337         9.796009     Better
21   AIMIRT       -5.785124       -8.976337         3.191213     Better
16    WHAIR       -6.040268       -8.976337         2.936068     Better
11    TFFIF      -11.688312       -8.976337        -2.711975      Worse
29      TVO      -12.566372       -8.976337        -3.590035      Worse
13      CPF      -14.634146       -8.976337        -5.657810      Worse
12    3BBIF      -17.283951       -8.976337        -8.307614      Worse
20      NER      -22.519685       -8.976337       -13.543348      Worse
5       DIF